# SDG 7 Progress Tracker — Satellite-Based Electrification Monitoring

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

This notebook implements the core SDG 7.1.1 monitoring framework using VIIRS nighttime
light (NTL) radiance as a direct proxy for electricity access — analogous to the approach
used by Echchabi et al. (2024) for SDG 6, but applied to energy with a stronger signal:
NTL *is* emitted light, not an indirect environmental correlate of piped infrastructure.

**Research question:** Are Brazil, China, and Morocco on track to achieve universal
electricity access (SDG 7.1.1 target: 95% by 2030)?

**Methods:**
1. NTL threshold classification → population-weighted access rate per year (2015–2023)
2. Theil-Sen slope + bootstrap 95% CI projection to 2030
3. SDG 7.1.1 gap analysis: On track / At risk / Off track
4. Composite Electrification Index (access rate + spatial equity)
5. Urban / rural disaggregation

**Key outputs:**
- `figures/sdg7_progress_timeline.png` — main publication figure
- `figures/sdg7_status_dashboard.png` — country status bar chart
- `figures/sdg7_country_comparison.png` — EI + Gini dual-axis
- `figures/sdg7_attainment_table.csv` — publishable results table

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sdg7_tracker import (
    classify_electrification, population_weighted_access_rate,
    access_rate_timeseries, theil_sen_projection, sdg7_gap_analysis,
    electrification_index, electrification_index_timeseries,
    urban_rural_disaggregation, plot_access_rate_timeseries,
    plot_sdg7_status_dashboard, plot_country_comparison,
    SDG7_UNIVERSAL_TARGET, NTL_ELECTRIFICATION_THRESHOLD,
)
from inequality import gini

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)
YEARS = list(range(2015, 2024))
print(f'SDG 7 Tracker loaded | Monitoring window: {YEARS[0]}–{YEARS[-1]}')
print(f'SDG 7.1.1 target: {SDG7_UNIVERSAL_TARGET*100:.0f}% by 2030')

## 1. Synthetic Spatially Correlated NTL Panel (2015–2023)

Parameters calibrated to match IEA/VIIRS empirical statistics for each country.
Morocco bbox extends to 20.8°N to include Western Sahara — a large, dark, sparsely
populated territory that depresses national NTL averages but has minimal population weight.

In [ ]:
def generate_temporal_panel(country, n_tiles, bbox, ntl_base, ntl_std,
                             mean_growth, growth_std, shocks, seed, years):
    """Generate tile-level yearly NTL panel with spatial autocorrelation."""
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    side = int(np.sqrt(n_tiles))
    xs = np.linspace(minx, maxx, side + 1)
    ys = np.linspace(miny, maxy, side + 1)
    geoms, tile_ids = [], []
    for i in range(side):
        for j in range(side):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f'{country[:3].upper()}_{i:02d}_{j:02d}')
    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl_base_tile = np.clip(rng.multivariate_normal(np.full(n, ntl_base), cov), 0.5, None)
    growth_cov = growth_std**2 * np.exp(-dists / (0.5 * (maxx - minx)))
    tile_growth = rng.multivariate_normal(np.full(n, mean_growth), growth_cov)
    pop = rng.lognormal(mean=np.log(100), sigma=1.2, size=n)

    rows = []
    ntl_curr = ntl_base_tile.copy()
    for yr in years:
        shock = shocks.get(yr, 0)
        ntl_curr = np.clip(ntl_curr + tile_growth + rng.normal(0, 0.3, n) + shock, 0, None)
        for i, (tid, g) in enumerate(zip(tile_ids, geoms)):
            rows.append({
                'tile_id': tid, 'country': country, 'year': yr,
                'ntl_mean': ntl_curr[i], 'pop_density': pop[i],
                'lon': centroids[i, 0], 'lat': centroids[i, 1],
            })
    return pd.DataFrame(rows), gpd.GeoDataFrame(
        {'tile_id': tile_ids, 'country': country, 'geometry': geoms},
        crs='EPSG:4326'
    )


# Parameters calibrated to IEA/VIIRS empirical data
country_params = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48,-23,-43,-18),
         ntl_base=9.5,  ntl_std=6.0,  mean_growth=0.42, growth_std=0.15,
         shocks={}, seed=1, years=YEARS),
    dict(country='China',   n_tiles=100, bbox=(116,29,122,33),
         ntl_base=21.0, ntl_std=12.0, mean_growth=1.85, growth_std=0.40,
         shocks={2020: -3.1}, seed=2, years=YEARS),   # COVID-19 dip
    dict(country='Morocco', n_tiles=100, bbox=(-17.1,20.8,-1.0,35.9),
         ntl_base=6.2,  ntl_std=4.0,  mean_growth=0.38, growth_std=0.12,
         shocks={2017: 1.5}, seed=3, years=YEARS),    # ONEE rural push
]

panel_dfs, geom_gdfs = [], {}
for params in country_params:
    df, gdf = generate_temporal_panel(**params)
    panel_dfs.append(df)
    geom_gdfs[params['country']] = gdf

panel_df = pd.concat(panel_dfs, ignore_index=True)
print(f'Panel: {len(panel_df):,} tile-year observations')
print(f'Countries: {panel_df.country.unique().tolist()}')
print(f'Years: {YEARS[0]}–{YEARS[-1]}')

## 2. NTL Threshold Sensitivity Analysis

The classification threshold is a key methodological choice. We compare three methods
for 2023 to justify using `method='fixed', threshold=2.0 nW/cm²/sr` as the primary estimate.

In [ ]:
snapshot_2023 = panel_df[panel_df.year == 2023].copy()

threshold_comparison = []
for country, grp in snapshot_2023.groupby('country'):
    ntl = grp.ntl_mean.values
    pop = grp.pop_density.values
    for method, thresh in [('Fixed (2.0)', 2.0), ('Fixed (1.0)', 1.0), ('Percentile 25th', None)]:
        if method == 'Percentile 25th':
            rate = population_weighted_access_rate(ntl, pop, method='percentile', percentile=25.0)
        else:
            rate = population_weighted_access_rate(ntl, pop, threshold=thresh)
        threshold_comparison.append({'Country': country, 'Method': method, 'Access Rate (%)': round(rate*100, 1)})

thresh_df = pd.DataFrame(threshold_comparison)
print(thresh_df.pivot(index='Country', columns='Method', values='Access Rate (%)').to_string())
print('\n→ Fixed threshold (2.0 nW/cm²/sr) selected for consistency with data_utils default.')

## 3. Population-Weighted Access Rates (2015–2023)

SDG 7.1.1 indicator: proportion of population with access to electricity.

In [ ]:
access_df = access_rate_timeseries(panel_df, threshold=NTL_ELECTRIFICATION_THRESHOLD)

pivot = access_df.pivot(index='country', columns='year', values='access_rate') * 100
print('Population-Weighted Electricity Access Rate (%) — NTL Model')
print('=' * 70)
print(pivot.round(1).to_string())
print(f'\nSDG 7.1.1 Target: {SDG7_UNIVERSAL_TARGET*100:.0f}%')

## 4. Theil-Sen Projections to 2030 with Bootstrap 95% CI

Non-parametric Theil-Sen slope used instead of OLS — robust to outliers and short time series.
Bootstrap resampling (n=1000) provides empirical confidence intervals.

In [ ]:
projection_results = {}
for country in ['Brazil', 'China', 'Morocco']:
    grp = access_df[access_df.country == country].sort_values('year')
    proj = theil_sen_projection(
        grp.year.values.astype(float),
        grp.access_rate.values,
        target_year=2030, n_bootstrap=1000,
    )
    projection_results[country] = proj
    print(f"{country:10s}: slope = {proj['annual_slope_pp']:+.3f} pp/yr | "
          f"2030 = {min(proj['projected_value'],1)*100:.1f}% "
          f"[{min(proj['ci_lower'],1)*100:.1f}%, {min(proj['ci_upper'],1)*100:.1f}%] | "
          f"trend: {proj['trend']} (p={proj['p_value']:.4f})")

## 5. SDG 7.1.1 Gap Analysis (Publishable Table 1)

In [ ]:
gap_df = sdg7_gap_analysis(access_df, target_year=2030, n_bootstrap=1000)

print('Table 1. SDG 7.1.1 Attainment Gap Analysis')
print('=' * 90)
print(gap_df.to_string(index=False))

gap_df.to_csv(FIGURES / 'sdg7_attainment_table.csv', index=False)
print('\nSaved: figures/sdg7_attainment_table.csv')

## 6. Publication Figure: SDG 7 Progress Timeline

In [ ]:
fig = plot_access_rate_timeseries(
    access_df, projection_results,
    target_line=SDG7_UNIVERSAL_TARGET,
    save_path=str(FIGURES / 'sdg7_progress_timeline.png'),
)
plt.show()

## 7. Country Status Dashboard

In [ ]:
fig = plot_sdg7_status_dashboard(
    gap_df,
    save_path=str(FIGURES / 'sdg7_status_dashboard.png'),
)
plt.show()

## 8. Composite Electrification Index (EI) — Access + Equity

EI = 0.5 × Access Rate + 0.5 × (1 − Gini)

A country with 90% access but extreme spatial inequality scores lower than
one with 85% access and equitable distribution. This is the gap vs SDG 10.

In [ ]:
ei_df = electrification_index_timeseries(panel_df)

ei_pivot = ei_df.pivot(index='country', columns='year', values='EI')
print('Composite Electrification Index (0–1, higher = better):')
print(ei_pivot.round(3).to_string())

fig = plot_country_comparison(
    ei_df,
    save_path=str(FIGURES / 'sdg7_country_comparison.png'),
)
plt.show()

## 9. Urban / Rural Disaggregation (2023 Snapshot)

In [ ]:
for country, grp in snapshot_2023.groupby('country'):
    gdf_snap = gpd.GeoDataFrame(grp, geometry=geom_gdfs[country].geometry.values, crs='EPSG:4326')
    ur = urban_rural_disaggregation(gdf_snap)
    print(f'\n{country} — Urban/Rural Disaggregation (2023):')
    print(ur.to_string(index=False))

## 10. Summary: Key Research Findings

In [ ]:
print('=' * 65)
print('KEY FINDINGS — SDG 7.1.1 TRACKING ANALYSIS')
print('=' * 65)

for _, row in gap_df.iterrows():
    status_icon = {'On track': '✓', 'At risk': '!', 'Off track': '✗'}.get(row['Status'], '?')
    print(f"\n[{status_icon}] {row['Country']}")
    print(f"    Access 2023 : {row['Access 2023 (%)']:.1f}%")
    print(f"    Projected 2030 : {row['Projected 2030 (%)']:.1f}% "
          f"[{row['CI Lower (%)']:.1f}–{row['CI Upper (%)']:.1f}%]")
    print(f"    Status : {row['Status']}")
    print(f"    Annual improvement : {row['Annual Slope (pp/yr)']:.2f} pp/yr")
    if row['Gap to Target (pp)'] > 0:
        print(f"    Gap to SDG target : {row['Gap to Target (pp)']:.1f} pp")

print()
print('Figures saved to figures/')